# 🤖 Machine Learning Interview Handbook

Этот ноутбук покрывает всю базу классического машинного обучения: от понимания метрик до устройства градиентного бустинга и полного цикла разработки (ML System Design).

---

## Блок 1: Типы задач и Метрики

### 📌 Типы задач с примерами:
- **Классификация:** Определение дискретного класса.
  * *Примеры:* Прогноз оттока клиентов (Churn), Выявление спама, Диагностика заболеваний.
- **Регрессия:** Предсказание непрерывного числа.
  * *Примеры:* Прогноз стоимости недвижимости, Предсказание температуры, Оценка времени прибытия такси (ETA).
- **Кластеризация:** Группировка объектов без учителя.
  * *Примеры:* Сегментация базы клиентов для маркетинга, Сжатие изображений.
- **Ранжирование:** Сортировка объектов по релевантности.
  * *Примеры:* Выдача поисковой системы, Рекомендации в Netflix/YouTube.

### 📊 Матрица ошибок (Confusion Matrix)
Для понимания метрик классификации важно знать компоненты матрицы:

| | Предсказано: **1** | Предсказано: **0** |
|---|---|---|
| **Актуально: 1** | **TP** (True Positive) | **FN** (False Negative) |
| **Актуально: 0** | **FP** (False Positive) | **TN** (True Negative) |

### 🔎 Метрики классификации (Deep Dive)
**1. Precision (Точность):** Доля реальных 1 среди всех, кого мы назвали 1. `TP / (TP + FP)`.
> *Кейс:* Если ложное срабатывание (FP) дорого стоит (например, блокировка нормальной транзакции).

**2. Recall (Полнота):** Какую долю реальных 1 мы нашли. `TP / (TP + FN)`.
> *Кейс (из лекций):* Госпиталь. Нам важно найти всех больных (Recall), даже если мы зря потревожим здоровых (ипохондриков).

**3. F1-score:** Среднее гармоническое Precision и Recall. Хорошо работает при дисбалансе классов.
   $$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

**4. ROC-AUC:** Площадь под кривой ошибок. Устойчива к дисбалансу, показывает качество ранжирования вероятностей моделью.

**Для регрессии:**
- **MAE:** Средний модуль ошибки (устойчива к выбросам).
- **MSE:** Средний квадрат ошибки (сильно штрафует за большие отклонения).
- **RMSE:** Корень из MSE (в тех же единицах, что и таргет).
- **R²:** Доля объясненной дисперсии (0.0 - 1.0).

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = [0, 1, 1, 0, 1, 0]
y_pred = [0, 1, 0, 0, 1, 1]

print(f"Accuracy: {accuracy_score(y_true, y_pred):.2f}")
print(f"Precision: {precision_score(y_true, y_pred):.2f}")
print(f"Recall: {recall_score(y_true, y_pred):.2f}")
print(f"F1: {f1_score(y_true, y_pred):.2f}")

Accuracy: 0.67
Precision: 0.67
Recall: 0.67
F1: 0.67


---
## Блок 2: Разделение данных (Train / Validation / Test)

Чтобы модель хорошо работала на реальных данных, их делят на части:
1. **Train (Обучающая):** На ней модель учит веса.
2. **Validation (Валидационная):** На ней мы подбираем гиперпараметры (например, глубину дерева) и выбираем лучшую архитектуру.
3. **Test (Тестовая):** "Золотой стандарт". Используется один раз в самом конце, чтобы оценить итоговое качество.

> **Зачем это нужно?** Если подбирать параметры на тестовой выборке, модель "подсмотрит" правильные ответы и оценка будет завышенной (**Data Leakage**).

---
## Блок 3: Переобучение и недообучение (Bias-Variance Tradeoff)

- **Underfitting (Недообучение):** Модель слишком простая (например, прямая линия для параболы). Высокий **Bias** (смещение). Она плохо работает и на обучении, и на тесте.
- **Overfitting (Переобучение):** Модель слишком сложная и заучила шум в данных. Высокий **Variance** (дисперсия). Она идеально работает на обучении, но проваливается на тесте.

**Как бороться?**
- **От переобучения:** Регуляризация (L1/L2), уменьшение сложности модели, больше данных, Dropout (в нейронках).
- **От недообучения:** Усложнение модели, добавление новых признаков, создание полиномиальных фичей.

---
## Блок 4: Дисбаланс классов и Stratified Split

**Дисбаланс данных:** Ситуация, когда одного класса намного больше (например, 99.9% транзакций — нормальные, 0.1% — фрод).
- **Проблема:** Accuracy будет 99.9%, даже если модель всегда предсказывает "нормально", но мы пропустим весь фрод.

**Stratified Split (Стратифицированное разбиение):**
При обычном случайном разбиении в тест может не попасть ни одного примера редкого класса. 
`stratify=y` в `train_test_split` гарантирует, что пропорции классов в Train и Test будут такими же, как в исходных данных.

In [10]:
from sklearn.model_selection import train_test_split
import numpy as np

X = np.random.rand(100, 5)
y = np.array([0]*90 + [1]*10) # Имитация сильного дисбаланса (90/10)

# Стратифицированное разбиение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Доля единиц в исходном y: {y.mean():.2f}")
print(f"Доля единиц в тренировочном y: {y_train.mean():.2f}") # Благодаря stratify=y, дисбаланс будет сохранен

Доля единиц в исходном y: 0.10
Доля единиц в тренировочном y: 0.10


---
## Блок 5: Валидация и Регуляризация (Борьба с переобучением)

### 🛡️ Регуляризация: L1 vs L2
Добавление штрафа к функции потерь, чтобы веса модели не росли бесконечно.

1. **L1 (Lasso) - $|w|$:** 
   - **Суть:** Добавляет сумму модулей весов.
   - **Эффект:** Обнуляет некоторые веса. Работает как **отбор признаков** (feature selection).
   - **Геометрия:** Область ограничений имеет вид ромба, углы которого чаще всего касаются линий уровня ошибки на осях (где веса = 0).

2. **L2 (Ridge) - $w^2$:**
   - **Суть:** Добавляет сумму квадратов весов.
   - **Эффект:** Уменьшает все веса равномерно, но почти никогда не обнуляет их. Борьба с **мультиколлинеарностью**.
   - **Геометрия:** Область ограничений - круг.

3. **ElasticNet:** Комбинация L1 и L2 штрафов.

In [11]:
import numpy as np
import joblib
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Создаем синтетические данные
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 2. Инициализируем модели
models = {
    'Logistic Regression': LogisticRegression(penalty='l2', C=1.0),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(learning_rate=0.1, max_depth=3, random_state=42)
}

# 3. Обучаем и оцениваем каждую модель в цикле
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    print(f"--- {name} ---")
    print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f}")
    print(classification_report(y_test, preds))

# 4. Сохранение лучшей модели (пример с RF)
best_model_name = 'Random Forest'
joblib.dump(models[best_model_name], 'final_model.pkl')
print(f"Model '{best_model_name}' saved to disk.")

# 5. Загрузка модели
loaded_model = joblib.load('final_model.pkl')
new_score = loaded_model.score(X_test, y_test)
print(f"Testing loaded model: Score = {new_score:.4f}")

--- Logistic Regression ---
ROC-AUC: 0.9339
              precision    recall  f1-score   support

           0       0.88      0.89      0.89       100
           1       0.89      0.88      0.88       100

    accuracy                           0.89       200
   macro avg       0.89      0.89      0.88       200
weighted avg       0.89      0.89      0.88       200

--- Random Forest ---
ROC-AUC: 0.9574
              precision    recall  f1-score   support

           0       0.89      0.92      0.91       100
           1       0.92      0.89      0.90       100

    accuracy                           0.91       200
   macro avg       0.91      0.91      0.90       200
weighted avg       0.91      0.91      0.90       200

--- Gradient Boosting ---
ROC-AUC: 0.9523
              precision    recall  f1-score   support

           0       0.88      0.95      0.91       100
           1       0.95      0.87      0.91       100

    accuracy                           0.91       200
   m

---
## 🛠️ Шпаргалка по классическим алгоритмам

| Алгоритм | Реализация (sklearn) | Плюсы | Минусы | Как бороться с переобучением |
|---|---|---|---|---|
| **k-NN** | `neighbors.KNeighborsClassifier` | Простой, не требует обучения | Медленный на больших данных, чувствителен к шуму | Увеличить `k` |
| **SVM** | `svm.SVC` | Хорош для малых данных и больших признаков | Медленный на больших выборках, трудно подбирать ядра | Уменьшать `C` |
| **Tree** | `tree.DecisionTreeClassifier` | Интерпретируемость, нет нужды в скалировании | Склонен к сильному переобучению | Ограничить `max_depth`, `min_samples_split` |
| **Random Forest** | `ensemble.RandomForestClassifier` | Устойчив к выбросам, параллельное обучение | Тяжелые модели, долгое инференс | Увеличить `n_estimators`, уменьшить `max_depth` |
| **Boosting** | `ensemble.GradientBoostingClassifier` | Высокая точность, исправляет ошибки | Склонен к переобучению, обучается последовательно | Ограничить `learning_rate` и `n_estimators` |


---
## Блок 6: Подбор гиперпараметров (Hyperparameter Tuning)

**Зачем это нужно?** В отличие от весов (которые модель учит сама), гиперпараметры (сила регуляризации `C`, глубина дерева `max_depth`) задает человек. От них критически зависит качество.

1. **GridSearchCV:** 
   - Перебирает **все** возможные комбинации параметров из заданной сетки.
   - **Плюсы:** Гарантированно находит лучший результат в рамках сетки.
   - **Минусы:** Очень медленно при большом количестве параметров.

2. **RandomizedSearchCV:** 
   - Выбирает **случайные** комбинации из сетки (количество попыток задается в `n_iter`).
   - **Плюсы:** Намного быстрее; часто находит хорошие параметры, пропуская заведомо плохие области.
   - **Минусы:** Может пропустить глобальный оптимум, если `n_iter` слишком мал.

In [12]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
import joblib

# Поиск оптимального гиперпараметра C (сила регуляризации)
params = {'C': [0.01, 0.1, 1.0, 10.0, 100.0]}
search = RandomizedSearchCV(LogisticRegression(), params, n_iter=3, cv=3, random_state=42)
search.fit(X_train, y_train)

print("Лучший найденный параметр C:", search.best_params_['C'])

# --- Сохранение модели на диск --- 
best_model = search.best_estimator_
joblib.dump(best_model, 'best_logreg_model.pkl')
print("Модель сохранена как best_logreg_model.pkl")

# Как потом загрузить (в продакшене):
# loaded_model = joblib.load('best_logreg_model.pkl')

Лучший найденный параметр C: 0.1
Модель сохранена как best_logreg_model.pkl


---
## Блок 7: Полный цикл ML задачи (ML System Design Lifecycle)

Классический вопрос на System Design интервью — "Опишите процесс решения ML-задачи от начала до конца":

1. **Бизнес-понимание:** Какую проблему мы решаем? Что будет метрикой успеха для бизнеса?
2. **Сбор и разметка данных (Data Collection):** Откуда берем данные? Хватает ли их качества? Как будем размечать (Толока, эксперты)?
3. **EDA и Препроцессинг:** Очистка от пропусков, удаление выбросов, лог-преобразования. 
4. **Генерация фичей (Feature Engineering):** Создание новых полезных колонок.
5. **Выбор модели и Обучение:** Начинаем с простых бейзлайнов (LogReg). Если бейзлайн работает - пробуем усложнять (CatBoost, LightGBM).
6. **Валидация (Evaluation):** Строгая проверка на `Validation` и `Test` сплитах. Анализ ошибок модели (почему она ошибается на конкретных примерах?).
7. **Деплой (Deployment):** Оборачивание модели в REST API микросервис (например, FastAPI, Flask + Docker).
8. **Мониторинг (Monitoring):** Модели со временем теряют актуальность (Data Drift / Concept Drift). Нужно мониторить "здоровье" модели и периодически ее дообучать.

### 📝 Контрольные вопросы для самопроверки перед интервью

1. Почему `Accuracy` нельзя использовать, если в выборке 99% нормальных транзакций и 1% мошеннических?
2. В чем фундаментальное отличие процесса построения деревьев в Random Forest и Gradient Boosting?
3. Как именно L1 регуляризация помогает отобрать только важные признаки?
4. Зачем в KNN обязательно выполнять масштабирование (StandardScaler) признаков перед обучением?

---
## 🎯 Реальные вопросы с собеседований (Middle Data Scientist)

### 🎤 Вопросы по моделям:
1. **Трансформация признаков:** Вы умножили все значения непрерывного признака (например, рост в см) на 100. Изменится ли от этого предсказание: а) Линейной регрессии? б) Дерева Решений? (Ответ обоснуйте математически).
2. **Смещение и дисперсия (Bias-Variance Tradeoff):** Если модель состоит из очень глубоких деревьев решений (без ограничения глубины), у нее высокое смещение или высокая дисперсия? Что произойдет, если мы объединим их в Random Forest?
3. **Предпосылки Логистической Регрессии:** Какие предположения о данных делает логистическая регрессия? (Ожидаемый ответ: отсутствие строгой мультиколлинеарности, линейная разделимость в пространстве признаков или трансформаций).
4. **Дисбаланс классов:** Метрика ROC-AUC показала 0.95 на тестовой выборке (классы 99 к 1). Можете ли вы сказать, что модель отлично ловит миноритарный класс? Почему здесь лучше смотреть на Precision-Recall AUC?

### 💻 Архитектурная задача (System Design ML):
Опишите пайплайн разработки рекомендательной системы фильмов. Какую архитектуру (двухэтапную: Candidate Retrieval + Ranking) вы бы использовали и какие метрики в оффлайне (NDCG) и в онлайне (A/B тест на кликабельность) вы бы выбрали?

### ✅ Ответы на классические вопросы ML

<details>
<summary><b>Ответ 1: Bias-Variance Tradeoff</b></summary>

- **Bias (Смещение):** Ошибка из-за того, что модель слишком простая (Underfitting) и не может описать данные.
- **Variance (Дисперсия):** Ошибка из-за того, что модель слишком сложная (Overfitting) и заучивает шум/выбросы.
- **Трейд-офф:** Идеальная модель находится на балансе. При усложнении Bias падает, но Variance растет (и наоборот).
</details>

<details>
<summary><b>Ответ 2: Random Forest vs Gradient Boosting</b></summary>

Оба ансамбли из деревьев, но:
- **Random Forest (Bagging):** Деревья строятся **параллельно** и независимо на случайных подвыборках данных. Хорошо борется с Variance (переобучением).
- **Gradient Boosting (Boosting):** Деревья строятся **последовательно**. Каждое следующее дерево исправляет ошибки предыдущего. Хорошо борется с Bias (недообучением). Более мощный, но склонен к переобучению.
</details>

<details>
<summary><b>Ответ 3: Метрики при дисбалансе</b></summary>

При дисбалансе классов (например, 99% здоровых, 1% больных) модель может предсказывать всем "Здоров" и получать Accuracy 99%.
В таких случаях смотрят на **Precision** (Точность, чтобы не ругаться впустую), **Recall** (Полнота, чтобы не пропустить больного), и объединяющую их метрику **F1-Score**. Для оценки алгоритма в целом — площадь под **PR Curve** или **ROC-AUC**.
</details>

---
## 🧠 Частые дополнительные вопросы (на понимание)

<details>
<summary><b>1. Разница между L1 (Lasso) и L2 (Ridge) регуляризацией</b></summary>

Обе штрафуют модель за слишком большие веса, чтобы избежать переобучения.
- **L1 (Lasso):** Использует модуль весов. Обладает свойством **отбора признаков** (зануляет веса неинформативных признаков).
- **L2 (Ridge):** Использует квадрат весов. Приближает веса к нулю, но редко зануляет полностью. Хорошо работает при сильной корреляции признаков.
</details>

<details>
<summary><b>2. Стратегии кросс-валидации</b></summary>

- **K-Fold:** Данные разбиваются на $K$ частей, модель обучается $K$ раз. Оценка надежна.
- **Stratified K-Fold:** Важен для задач классификации с дисбалансом. Камера следит за тем, чтобы соотношение классов во всех фолдах было одинаковым.
- **Time Series Split:** Использовать обычный K-fold для временных рядов нельзя (нельзя тестировать модель "в прошлом", обучив на "будущем"). Разделение делается так, чтобы обучаться на более старых данных, а тестироваться на более новых.
</details>

---
### 📝 Практические задачи для самопроверки

1. **Метрики:** Рассчитайте Precision и Recall вручную, если TP=40, FP=10, FN=20.
2. **Регуляризация:** Какую регуляризацию вы выберете, если у вас 1000 признаков, но вы подозреваете, что реально важны только 10?
3. **Sklearn:** Обучите `RandomForestClassifier` на встроенном датасете `iris` и выведите `feature_importances_`.
4. **Serialization:** Сохраните обученную модель `DecisionTreeClassifier` в файл и загрузите обратно.

<details>
<summary>✅ Посмотреть ответы</summary>

1. **Precision** = 40/(40+10) = 0.8; **Recall** = 40/(40+20) = 0.67.
2. **L1 (Lasso)**, так как она обнуляет веса неважных признаков.
3. Юзкейс: `model.fit(X, y); print(model.feature_importances_)`.
4. Юзкейс: `joblib.dump(model, 'tree.pkl')` -> `model = joblib.load('tree.pkl')`.

</details>